# Detekcja sytuacji do chamowania

### Importy

In [1]:
import carla
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort
import cv2
import os
import numpy as np
import random
import clip

import time
from pathlib import Path
from typing import List, Tuple, Optional
import pygame
import torch
import sys

import math
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
from collections import deque
import supervision as sv
import logging

pygame 2.6.1 (SDL 2.28.4, Python 3.10.8)
Hello from the pygame community. https://www.pygame.org/contribute.html


#### Przygotowanie logowania

In [2]:
class LevelFilter(logging.Filter):
    def __init__(self, level):
        super().__init__()
        self.level = level

    def filter(self, record):
        return record.levelno == self.level

In [3]:
logger = logging.getLogger("AEB")
logger.setLevel(logging.DEBUG)
logger.propagate = False

formatter = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")

critical_handler = logging.FileHandler("aeb_critical.log", encoding="utf-8")
critical_handler.setLevel(logging.CRITICAL)
critical_handler.setFormatter(formatter)

warning_handler = logging.FileHandler("aeb_warning.log", encoding="utf-8")
warning_handler.setLevel(logging.WARNING)
warning_handler.setFormatter(formatter)

critical_handler.addFilter(LevelFilter(logging.CRITICAL))
warning_handler.addFilter(LevelFilter(logging.WARNING))

logger.addHandler(critical_handler)
logger.addHandler(warning_handler)

## Inicjowanie symulatora CARLA

In [4]:
client = carla.Client("localhost", 2000)
tm = client.get_trafficmanager(8000)
TM_PORT = tm.get_port()

world = client.get_world()
spawn_points = world.get_map().get_spawn_points()

vehicle_bp = world.get_blueprint_library().filter("grandtourer")
start_pos = spawn_points[0]

In [ ]:
sys.path.append("./CARLA_0.9.15/WindowsNoEditor/PythonAPI/carla")

from agents.navigation.global_route_planner import GlobalRoutePlanner

### Stowrzenie pojazdu

In [6]:
point_a = carla.Location(x=-64.599998, y=24.500000, z=1.000000)
point_b = carla.Location(x=-113.9, y=14.4, z=1)

In [7]:
vehicle = None

offsets = [(0, 0), (1, 0), (-1, 0), (0, 1), (0, -1), (2, 0), (-2, 0), (0, 2), (0, -2)]

for dx, dy in offsets:
    loc = carla.Location(x=point_a.x + dx, y=point_a.y + dy, z=point_a.z)
    spawn_transform = carla.Transform(loc, carla.Rotation(yaw=0))
    try:
        vehicle = world.spawn_actor(vehicle_bp[0], spawn_transform)
        print(f"Vehicle spawned at: {spawn_transform.location}")
        break
    except RuntimeError:
        continue

if vehicle is None:
    print(
        "Nie udało się spawnować pojazdu – spróbuj innego punktu startowego lub usuń przeszkody"
    )

Vehicle spawned at: Location(x=-64.599998, y=24.500000, z=1.000000)


#### Konfiguracja Traffic Managera

In [8]:
vehicle.set_autopilot(True, TM_PORT)

In [ ]:
tm.ignore_lights_percentage(vehicle, 100.0)
tm.ignore_vehicles_percentage(vehicle, 100.0)
tm.ignore_walkers_percentage(vehicle, 100.0)
tm.distance_to_leading_vehicle(vehicle, 0.0)
tm.update_vehicle_lights(vehicle, True)

### Stworzenie kamery

In [10]:
CAMERA_POS_Z = 1.5
CAMERA_POS_X = 1

camera_bp = world.get_blueprint_library().find("sensor.camera.rgb")
camera_bp.set_attribute("image_size_x", "640")
camera_bp.set_attribute("image_size_y", "640")

camera_init_trans = carla.Transform(carla.Location(z=CAMERA_POS_Z, x=CAMERA_POS_X))
camera = world.spawn_actor(camera_bp, camera_init_trans, attach_to=vehicle)


def camera_callback(image, data_dict):
    data_dict["image"] = np.reshape(
        np.copy(image.raw_data), (image.height, image.width, 4)
    )[:, :, :3]


image_w = camera_bp.get_attribute("image_size_x").as_int()
image_h = camera_bp.get_attribute("image_size_y").as_int()

camera_data = {"image": np.zeros((image_h, image_w))}
camera.listen(lambda image: camera_callback(image, camera_data))

## Stałe

In [11]:
YOLO_DETEC_WEIGHTS = "runs/detect/yolov11l-bdd100k/weights/best.pt"
YOLO_LIGHT_WEIGHTS = "runs/detect/yolov11m-light/weights/best.pt"

CONF_THRES = 0.3

CLASSES = None

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

bytetracker_kwargs = dict(
    lost_track_buffer=50,
    minimum_consecutive_frames=2,
    minimum_matching_threshold=0.4,
    track_activation_threshold=0.3,
)

In [40]:
BUFFLEN = 10  # ile próbek prędkości trzymamy (jak miałeś 10)
TTC_WARNING_THRESHOLD = 3.0  # [s] czas do kolizji przy którym dajemy ostrzeżenie
TTC_BRAKE_THRESHOLD = (
    1.0  # [s] czas do kolizji przy którym zaczynamy hamowanie krytyczne
)
CONF_WARN_THRESHOLD = 50.0  # [%] powyżej czego = warning
CONF_CRIT_THRESHOLD = 60.0  # [%] powyżej czego = critical
A_REF = 10.0  # skalowanie przyspieszenia (m/s^2) -> referencyjna "duża" wartość
MIN_APPROACH_SPEED = (
    0.3  # [m/s] przy mniejszej prędkości TTC traktujemy jako nieskończone
)
W_GROWTH = 0.4
W_DIST = 0.42
W_ALIGN = 0.65
COS_ANGLE = 0.5
H, W = (640, 640)
EPS = H * 0.001  # 0.1% szerokości obrazu
CONF_WEIGHTS = {"ttc": 0.10, "accel": 0.3, "brake": 0.0, "traj": 0.6, "close": 1.0}
MAX_WIDTH_RATIO = 0.57
MIN_WIDTH_RATIO = 0.02
TRAPEZOID_HEIGHT_RATIO = 0.43
CLOSE_HEIGHT = 0.35
CLOSE_WIDT_MODIFIER = 0.7

MIN_DIST = 1
MAX_DIST = 200.0

In [13]:
FOCAL_LENGTH_PX = (640 / 2) / math.tan(math.radians(90) / 2)
#             ['bike', 'bus', 'car', 'motor', 'person', 'rider', 'traffic light', 'traffic sign', 'train', 'truck']
REAL_HEIGHT = [1.1, 3.5, 1.5, 1.15, 1.7, 1.83, 0.9, 0.6, 3.5, 3.5]

In [ ]:
MAX_BREAK = 0.4
ALPHA = 0.65

## Funkcje pomocnicze

### Funkcja hamowania

In [15]:
def emergency_brake(vehicle, brake_value=MAX_BREAK):
    # control = carla.VehicleControl()
    control = vehicle.get_control()
    control.throttle = 0.0
    brake = min(max(brake_value, 0.0), 1.0)
    control.brake = float(ALPHA * brake + (1 - ALPHA) * control.brake)
    vehicle.apply_control(control)

In [16]:
def break_handler(vehicle, level, confidence, autopilot_active, last_level):
    if level == "critical" and last_level == "critical":
        break_value = 0.2 + 0.4 * (confidence - CONF_CRIT_THRESHOLD) / (
            100 - CONF_CRIT_THRESHOLD
        )
        if autopilot_active:
            vehicle.set_autopilot(False, TM_PORT)
            autopilot_active = False
        emergency_brake(vehicle, brake_value=break_value)
    else:
        if not autopilot_active:
            vehicle.set_autopilot(True, TM_PORT)
            autopilot_active = True
    return autopilot_active

### Funkcje prędkości

In [17]:
def estimate_speed(track_id, cx, cy, fps, prev_positions, ppm=None, car_speed=None):
    time_const = 1.0 / fps
    speed = None

    if track_id in prev_positions:
        prev_x, prev_y, _ = prev_positions[track_id]
        delta_d = math.hypot(cx - prev_x, cy - prev_y)
        if not ppm and car_speed and car_speed > 0:
            speed_m_per_s = car_speed / 3.6
            ppm = delta_d / (speed_m_per_s * time_const)
        if ppm:
            speed = (delta_d / ppm) / time_const * 3.6  # km/h

    return speed, ppm

In [18]:
def update_speed_buffer(track_id, speed, speed_buffers, maxlen=BUFFLEN):
    if track_id not in speed_buffers:
        speed_buffers[track_id] = deque(maxlen=maxlen)
    speed_buffers[track_id].append(speed)

In [19]:
def calculate_acceleration(track_id, speed_buffers, beta=0.0625, maxlen=BUFFLEN):
    acceleration = None
    speeds = [s for s in speed_buffers[track_id] if s is not None]
    currentlen = len(speeds)
    halflen = currentlen // 2
    if (
        track_id in speed_buffers
        and currentlen > maxlen // 2
        and speeds[-1] is not None
        and speeds[-2] is not None
    ):
        if not speeds or (speeds[-1] is None and speeds[-2] is None):
            return 0.0
        initial_avg = sum(speeds[:halflen]) / halflen
        final_avg = sum(speeds[halflen:]) / (maxlen - halflen)
        acceleration = (final_avg - initial_avg) / (currentlen * beta)
    return acceleration

In [20]:
def cleanup_old_tracks(
    prev_positions, speed_buffers, last_seen, last_levels, sizes, max_age_seconds=10
):
    now = time.time()
    to_delete = [tid for tid, last in last_seen.items() if now - last > max_age_seconds]
    for tid in to_delete:
        prev_positions.pop(tid, None)
        speed_buffers.pop(tid, None)
        last_seen.pop(tid, None)
        sizes.pop(tid, None)
        last_levels.pop(tid, None)

In [21]:
def estimate_distance_from_bbox(
    y1,
    y2,
    bbox_height_px,
    real_height_m,
    focal_length_px,
    camera_height_m=None,
    pitch_rad=0.0,
    frame_height=None,
):
    if (
        bbox_height_px is None
        or bbox_height_px <= 0
        or real_height_m is None
        or focal_length_px is None
    ):
        return None

    est_Z = (focal_length_px * real_height_m) / bbox_height_px

    if camera_height_m is not None and frame_height is not None:
        est_Z = est_Z * np.cos(pitch_rad)

    est_Z_clamped = np.clip(est_Z, MIN_DIST, MAX_DIST)

    return est_Z_clamped

### Określanie poziomu niebezpieczeństwa

In [ ]:
def calculate_confidence_score(
    accel_m_s2,
    approach_speed_mps,
    distance_m,
    brake_on=False,
    cx=None,
    cy=None,
    y2=None,
    current_size=None,
    last_size=None,
    frame_width=None,
    frame_height=None,
    delta_x=0.0,
    delta_y=0.0,
    delta_y2=0.0,
):
    # --- 1. TTC component ---
    ttc = None
    ttc_score = 0.0
    if (
        distance_m is not None
        and approach_speed_mps is not None
        and approach_speed_mps > MIN_APPROACH_SPEED
        and last_size is not None
        and (last_size < current_size or delta_y > 0)
    ):
        ttc = distance_m / approach_speed_mps
        if ttc <= 0:
            ttc_score = 1.0
        else:
            if ttc <= TTC_BRAKE_THRESHOLD:
                ttc_score = 1.0
            elif ttc < TTC_WARNING_THRESHOLD:
                ttc_score = np.clip(
                    (TTC_WARNING_THRESHOLD - ttc)
                    / (TTC_WARNING_THRESHOLD - TTC_BRAKE_THRESHOLD),
                    0.0,
                    1.0,
                )

    accel_score = 0.0
    if accel_m_s2 is not None and (
        approach_speed_mps is not None and approach_speed_mps > MIN_APPROACH_SPEED
    ):
        accel_effect = abs(accel_m_s2)
        accel_score = 1.0 - np.exp(-accel_effect / A_REF)

    brake_score = 1.0 if brake_on else 0.0

    growth_score = 0.0
    traj_score = 0.0
    align_score = 0.0
    dist_score = 0.0
    growth = 0.0
    close_score = 0.0
    close_within_lane = None
    within_lane = None

    frame_area = frame_width * frame_height
    if last_size is not None and last_size > 0:
        # growth = max(current_size - last_size, 0) / last_size
        growth = np.log1p(max(current_size - last_size, 0)) / np.log1p(last_size)
        growth_score = np.clip(growth, 0.0, 1.0)

    if None not in [frame_width, frame_height, cx, cy, y2]:
        target_x = frame_width / 2.0
        target_y = frame_height

        vec_motion = np.array([delta_x, delta_y2])
        vec_target = np.array([target_x - cx, target_y - y2])
        cos_angle = np.dot(vec_motion, vec_target) / (
            np.linalg.norm(vec_motion) * np.linalg.norm(vec_target) + 1e-6
        )

        vertical_dist = frame_height - y2
        top_y = int(frame_height * TRAPEZOID_HEIGHT_RATIO)
        width_ratio = MIN_WIDTH_RATIO + (MAX_WIDTH_RATIO - MIN_WIDTH_RATIO) * (
            (cy - top_y) / (frame_height - top_y)
        )
        lane_half_width = frame_width * width_ratio
        within_lane = (
            abs(cx - target_x) <= lane_half_width and cy > top_y and vertical_dist > EPS
        )

        moving_towards = delta_y > EPS and cos_angle > COS_ANGLE

        align_score = np.clip((cos_angle - COS_ANGLE) / (1 - COS_ANGLE), 0.0, 1.0)
        dist_modifier = CLOSE_HEIGHT * frame_height * 3 / 4
        dist_score = 1.0 - np.clip((vertical_dist - dist_modifier) / top_y, 0.0, 1.0)

        if within_lane or moving_towards:
            traj_score = np.clip(
                W_GROWTH * growth_score + W_DIST * dist_score + W_ALIGN * align_score,
                0.0,
                1.0,
            )

        close_width_ratio = MIN_WIDTH_RATIO + (MAX_WIDTH_RATIO - MIN_WIDTH_RATIO) * (
            (y2 - top_y) / (frame_height - top_y)
        )
        close_half_width = frame_width * close_width_ratio * CLOSE_WIDT_MODIFIER
        close_within_lane = abs(cx - target_x) <= close_half_width and y2 > top_y

        if current_size is not None:
            if current_size >= 0.3 * frame_area:
                close_score = 0.25
        if (
            vertical_dist < frame_height * CLOSE_HEIGHT
            and close_within_lane
            and vertical_dist > EPS * 10
        ):
            close_score = 1.0

    final_score = (
        CONF_WEIGHTS["ttc"] * ttc_score
        + CONF_WEIGHTS["accel"] * accel_score
        + CONF_WEIGHTS["brake"] * brake_score
        + CONF_WEIGHTS["traj"] * traj_score
        + CONF_WEIGHTS["close"] * close_score
    )

    confidence = np.clip(final_score * 100.0, 0.0, 100.0)

    if confidence >= CONF_CRIT_THRESHOLD:
        level = "critical"
    elif confidence >= CONF_WARN_THRESHOLD:
        level = "warning"
    else:
        level = "safe"

    scores = {
        "ttc": ttc_score,
        "accel": accel_score,
        "brake": brake_score,
        "traj": {
            "full": traj_score,
            "growht": growth_score,
            "dist": dist_score,
            "align": align_score,
        },
        "close": close_score,
    }

    return confidence, level, scores

### Logowanie poziomów niebezpieczeństwa 

In [23]:
def fmt(value, digits=2):
    return f"{value:.{digits}f}" if value is not None else "N/A"

In [24]:
def log_if_critical(
    tracker_id,
    conf,
    level,
    last_levels,
    scores,
    distance_m,
    speed,
    acceleration,
    ttc=None,
):
    info = (
        f"[TRACK {tracker_id}] {level.upper()} EVENT | "
        f"Break={vehicle.get_control().brake}, "
        f"Conf={fmt(conf)}, Dist={fmt(distance_m)}m, Speed={fmt(speed)}m/s, "
        f"Accel={fmt(acceleration)}m/s², Scores={scores}, TTC={fmt(ttc)}"
    )
    if level == "critical" and last_levels == "critical":
        logger.critical(info)
    elif level == "warning" and last_levels == "warning":
        logger.warning(info)

### Funkcje pomocnicze do modeli

In [25]:
def run_detection(frame, model, conf=CONF_THRES, classes=CLASSES, device=DEVICE):
    return model.predict(
        frame, conf=conf, classes=classes, device=device, verbose=False
    )[0]

In [26]:
def filter_lights(results):
    detections = []
    for box in results.boxes:
        cls_id = int(box.cls[0])
        if cls_id:
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            conf = float(box.conf[0])
            detections.append(([x1, y1, x2 - x1, y2 - y1], conf, cls_id))
    return detections

### Funkcje rysowania

In [27]:
def draw_lights(detections, frame):
    color = (250, 0, 0)
    for (x, y, w_box, h_box), *_ in detections:
        cv2.rectangle(frame, (x, y), (x + w_box, y + h_box), color, 4)
    return frame

In [ ]:
def draw_track_info(
    frame,
    track_id,
    x1,
    y1,
    x2,
    y2,
    color,
    frame_width,
    frame_height,
    speed=None,
    acceleration=None,
    conf=None,
    level=None,
    scores=None,
    show_bar=True,
):
    if level is not None:
        if level == "critical":
            final_color = (256, 0, 0)
        elif level == "warning":
            final_color = (256, 256, 0)
        else:
            final_color = (0, 256, 0)
    else:
        final_color = color

    cv2.rectangle(frame, (x1, y1), (x2, y2), final_color, 4)

    text = ""
    if speed is not None:
        text += f"{speed:.1f} km/h"
    if acceleration is not None:
        text += f" | a:{acceleration:.2f}"

    text_scale, text_thickness = 0.5, 2
    text_size = cv2.getTextSize(
        text, cv2.FONT_HERSHEY_SIMPLEX, text_scale, text_thickness
    )[0]

    bg_x1, bg_y1 = x1, max(0, y1 - text_size[1] - 10)
    bg_x2, bg_y2 = x1 + text_size[0] + 5, y1 - 10

    if bg_y1 >= 0 and bg_y2 < frame_height and bg_x2 < frame_width:
        cv2.rectangle(frame, (bg_x1, bg_y1), (bg_x2, bg_y2), (255, 255, 255), -1)
        cv2.putText(
            frame,
            text,
            (x1, y1 - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            text_scale,
            color,
            text_thickness,
        )

    return frame

In [29]:
def draw_frame_pygame(screen, frame):
    frame_surface = pygame.surfarray.make_surface(np.transpose(frame, (1, 0, 2)))
    screen.blit(frame_surface, (0, 0))
    pygame.display.flip()

In [ ]:
def draw_collision_cone(frame, frame_width, frame_height, color=(0, 255, 255)):
    target_x = frame_width // 2

    bottom_left = (int(target_x - frame_width * MAX_WIDTH_RATIO), frame_height)
    bottom_right = (int(target_x + frame_width * MAX_WIDTH_RATIO), frame_height)

    top_y = int(frame_height * TRAPEZOID_HEIGHT_RATIO)
    top_left = (int(target_x - frame_width * MIN_WIDTH_RATIO), top_y)
    top_right = (int(target_x + frame_width * MIN_WIDTH_RATIO), top_y)

    cone_pts = np.array(
        [bottom_left, bottom_right, top_right, top_left], dtype=np.int32
    )

    overlay = frame.copy()
    cv2.fillPoly(overlay, [cone_pts], color) 
    frame = cv2.addWeighted(overlay, 0.2, frame, 0.8, 0) 

    cv2.polylines(frame, [cone_pts], isClosed=True, color=color, thickness=2)

    close_y = int(frame_height * (1.0 - CLOSE_HEIGHT)) 

    cv2.line(frame, (0, close_y), (frame_width, close_y), (0, 0, 255), 2)

    top_half_width = int(frame_width * MIN_WIDTH_RATIO * CLOSE_WIDT_MODIFIER)
    top_left2 = (int(target_x - top_half_width), top_y)
    top_right2 = (int(target_x + top_half_width), top_y)

    bottom_half_width = int(frame_width * MAX_WIDTH_RATIO * CLOSE_WIDT_MODIFIER)
    bottom_left2 = (int(target_x - bottom_half_width), frame_height)
    bottom_right2 = (int(target_x + bottom_half_width), frame_height)

    pts = np.array([top_left2, top_right2, bottom_right2, bottom_left2], np.int32)
    cv2.polylines(frame, [pts], isClosed=True, color=(0, 0, 255), thickness=2)

    return frame

### Funkcja do przetworzenia detekcji 

In [ ]:
def update_and_draw_tracks(
    tracker,
    detections,
    frame,
    frame_width,
    frame_height,
    fps,
    prev_positions,
    speed_buffers,
    last_seen,
    last_size,
    last_level,
    last_levels,
    autopilot_active,
    car_speed=None,
    ppm=20,
    beta=0.0625,
    brake_on=False,
    max_age_seconds=10,
):
    tracked_detections = tracker.update_with_detections(detections)
    max_conf = 0
    max_level = None

    for i in range(len(tracked_detections.xyxy)):
        xyxy = tracked_detections.xyxy[i]
        confidence = tracked_detections.confidence[i]
        class_id = tracked_detections.class_id[i]
        tracker_id = tracked_detections.tracker_id[i]

        x1, y1, x2, y2 = map(int, xyxy)
        cx, cy = (x1 + x2) // 2, (y1 + y2) // 2

        color = (
            random.randint(50, 200),
            random.randint(50, 200),
            random.randint(50, 200),
        )

        if tracker_id in prev_positions:
            prev_x, prev_y, prev_y2 = prev_positions[tracker_id]
            delta_x = cx - prev_x
            delta_y = cy - prev_y
            delta_y2 = y2 - prev_y2
        else:
            delta_x = 0.0
            delta_y = 0.0
            delta_y2 = 0.0

        if class_id in [6, 7]:
            speed, ppm = estimate_speed(
                tracker_id, cx, cy, fps, prev_positions, ppm=None, car_speed=car_speed
            )
        else:
            speed, _ = estimate_speed(tracker_id, cx, cy, fps, prev_positions, ppm=ppm)

        update_speed_buffer(tracker_id, speed, speed_buffers)
        acceleration = calculate_acceleration(tracker_id, speed_buffers, beta)

        distance_m = estimate_distance_from_bbox(
            y1,
            y2,
            bbox_height_px=(y2 - y1),
            real_height_m=REAL_HEIGHT[class_id],
            focal_length_px=FOCAL_LENGTH_PX,
            camera_height_m=CAMERA_POS_Z,
            frame_height=frame_height,
        )

        conf, level, scores = calculate_confidence_score(
            acceleration,
            speed,
            distance_m,
            brake_on=brake_on,
            cx=cx,
            cy=cy,
            y2=y2,
            current_size=abs(x1 - x2) * abs(y1 - y2),
            last_size=last_size.get(tracker_id, None),
            frame_width=frame_width,
            frame_height=frame_height,
            delta_x=delta_x,
            delta_y=delta_y,
            delta_y2=delta_y2,
        )

        if max_conf < conf:
            max_conf = conf
            max_level = level

        log_if_critical(
            tracker_id,
            conf,
            level,
            last_levels.get(tracker_id, None),
            scores,
            distance_m,
            speed,
            acceleration,
            ttc=distance_m / speed if (distance_m and speed and speed > 0) else None,
        )

        frame = draw_track_info(
            frame,
            tracker_id,
            x1,
            y1,
            x2,
            y2,
            color,
            frame_width,
            frame_height,
            speed,
            acceleration,
            conf=conf,
            level=level,
            scores=scores,
        )

        last_seen[tracker_id] = time.time()
        prev_positions[tracker_id] = (cx, cy, y2)
        last_size[tracker_id] = abs(x1 - x2) * abs(y1 - y2)
        last_levels[tracker_id] = level

    autopilot_active = break_handler(
        vehicle, max_level, max_conf, autopilot_active, last_level
    )
    cleanup_old_tracks(
        prev_positions,
        speed_buffers,
        last_seen,
        last_size,
        last_levels,
        max_age_seconds,
    )
    # frame = draw_collision_cone(frame, frame_width, frame_height)

    return frame, autopilot_active, max_level

## Główna pętla

#### Łądowanie modeli i słowników

In [ ]:

model = YOLO(YOLO_DETEC_WEIGHTS)
light = YOLO(YOLO_LIGHT_WEIGHTS)

In [ ]:

tracker = sv.ByteTrack()  # **bytetracker_kwargs)

In [34]:
prev_positions = {}
speed_buffers = {}
last_seen = {}
last_size = {}
last_levels = {}
autopilot_active = True

#### Pętla

In [ ]:
quit = False
last_level = None

pygame.init()
clock = pygame.time.Clock()
screen = pygame.display.set_mode((H, W))
font = pygame.font.SysFont(None, 24)

while not quit:
    world.tick()
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            quit = True
        if event.type == pygame.KEYDOWN:
            if event.key == pygame.K_ESCAPE:
                quit = True

    v = vehicle.get_velocity()
    speed = round(3.6 * math.sqrt(v.x**2 + v.y**2 + v.z**2), 0)

    image = camera_data["image"].copy()
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    surface = pygame.surfarray.make_surface(image.swapaxes(0, 1))
    fps = clock.get_fps() or 30
    tracker.frame_rate = fps

    lights = run_detection(image, light)
    lights_detections = filter_lights(lights)
    frame = draw_lights(lights_detections, image)

    results = run_detection(image, model)
    boxes = results.boxes

    detections = sv.Detections.from_ultralytics(results)
    frame, autopilot_active, last_level = update_and_draw_tracks(
        tracker,
        detections,
        image,
        W,
        H,
        fps,
        prev_positions,
        speed_buffers,
        last_seen,
        last_size,
        last_level,
        last_levels,
        autopilot_active,
        car_speed=speed,
        brake_on=len(lights_detections) > 0,
    )

    draw_frame_pygame(screen, frame)

    text = font.render(f"Speed: {int(speed)} km/h", True, (255, 255, 255))
    screen.blit(text, (10, 10))

    text_brake_actual = font.render(
        f"Vehicle Brake: {vehicle.get_control().brake:.2f}", True, (255, 100, 100)
    )
    screen.blit(text_brake_actual, (10, 60))

    pygame.display.flip()
    clock.tick(60)

pygame.quit()

#### Zmiana autopilota

In [ ]:
vehicle.set_autopilot(False, TM_PORT)

### Clean vehicle

In [ ]:
camera.stop()
if vehicle is not None:
    vehicle.destroy()
    vehicle = None

RuntimeError: trying to operate on a destroyed actor; an actor's function was called, but the actor is already destroyed.